# Electrolyser Telemetry — Fault Detection & Classification Pipeline

**Research-grade ML pipeline** for predictive maintenance of PEM electrolyser stacks.
Three progressive models demonstrate the full spectrum from deployment-ready
unsupervised detection to high-accuracy supervised classification with temporal
context and model interpretability.

| # | Model | Paradigm | Rationale |
|---|---|---|---|
| 1 | **Isolation Forest** | Unsupervised | No labels needed; catches novel/zero-day faults |
| 2 | **Random Forest** | Supervised multiclass | Fast, interpretable baseline with feature importance |
| 3 | **LSTM Autoencoder** | Temporal unsupervised | Exploits sequential degradation trajectories; detects early drift |

All three are evaluated against the same held-out test set with identical metrics
(precision, recall, F1, AUC-ROC) to allow direct comparison.

### Pre-requisites
```bash
pip install pandas pyarrow scikit-learn matplotlib seaborn torch shap
# Collect labelled data (run simulator + inject faults during collection):
python scripts/export_dataset.py --duration 600 --out data/telemetry_labelled.parquet
```

### Dataset generation note
For a representative dataset, run the simulator for ≥600 s while cycling through
all 15 fault modes using `scripts/test_faults.py`. This produces ~1200 labelled
rows per electrolyser (600 normal + ~40 per fault class).

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import pathlib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    ConfusionMatrixDisplay, precision_recall_fscore_support
)
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import shap

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DATA_PATH = pathlib.Path('../data/telemetry_labelled.parquet')
FIG_DIR   = pathlib.Path('../data')
FIG_DIR.mkdir(exist_ok=True)
print('PyTorch:', torch.__version__, '  CUDA:', torch.cuda.is_available())
print('Data path:', DATA_PATH.resolve())

## 1. Data Loading & Exploratory Analysis

In [ ]:
df = pd.read_parquet(DATA_PATH)
df['ts'] = pd.to_datetime(df['timestamp'], unit='s')
df = df.sort_values('ts').reset_index(drop=True)

print(f'Shape: {df.shape}  |  Columns: {list(df.columns)}')
print(f'Time span: {df["ts"].min()}  →  {df["ts"].max()}')
print(f'\nLabel distribution:')
print(df['fault_label'].value_counts().to_string())
print(f'\nFault rate: {df["is_fault"].mean():.1%}')

In [ ]:
# Correlation heatmap on raw features
CELL_COLS   = [f'cell_{i}_voltage' for i in range(1,6)]
STACK_COLS  = ['stack_current','stack_temperature','stack_pressure']
GAS_COLS    = ['h2_flow_rate','o2_flow_rate']
OTHER_COLS  = ['tank_pressure','water_flow','irradiance_1','irradiance_2']
RAW_FEATURES = CELL_COLS + STACK_COLS + GAS_COLS + OTHER_COLS

fig, ax = plt.subplots(figsize=(14,10))
corr = df[RAW_FEATURES].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.3, ax=ax, annot_kws={'size':7})
ax.set_title('Feature Correlation Matrix — Normal Operating Conditions', pad=12)
plt.tight_layout()
plt.savefig(FIG_DIR/'correlation_heatmap.png', dpi=150)
plt.show()

In [ ]:
# Per-fault box plots for key discriminating features
DISC_FEATURES = ['cell_1_voltage','h2_flow_rate','water_flow','stack_current','tank_pressure']
fig, axes = plt.subplots(1, len(DISC_FEATURES), figsize=(18, 5), sharey=False)
for ax, feat in zip(axes, DISC_FEATURES):
    order = df.groupby('fault_label')[feat].median().sort_values().index
    sns.boxplot(data=df, x=feat, y='fault_label', order=order, ax=ax,
                orient='h', palette='muted', linewidth=0.8, fliersize=2)
    ax.set_title(feat.replace('_',' ').title(), fontsize=10)
    ax.set_xlabel('')
    ax.set_ylabel('')
axes[0].set_ylabel('Fault Class')
fig.suptitle('Feature Distributions Across Fault Classes', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR/'fault_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Feature Engineering

In [ ]:
# ── Raw features ──────────────────────────────────────────────────────────────
df['cell_voltage_max']  = df[CELL_COLS].max(axis=1)
df['cell_voltage_min']  = df[CELL_COLS].min(axis=1)
df['cell_voltage_std']  = df[CELL_COLS].std(axis=1)
df['cell_voltage_range']= df['cell_voltage_max'] - df['cell_voltage_min']

# ── Domain-expert derived features ───────────────────────────────────────────
# H₂/O₂ stoichiometric ratio — normal ≈ 2.0; deviates under gas crossover
df['h2_o2_ratio']        = df['h2_flow_rate'] / (df['o2_flow_rate'] + 1e-9)
# Electrochemical efficiency proxy: H₂ yield per unit current
df['faradaic_efficiency']= df['h2_flow_rate'] / (df['stack_current'] + 1e-9)
# Stack power (W) — diagnostic for DC-DC failures
df['stack_power']        = df['stack_current'] * df['cell_voltage_max'] * 5
# Thermal load index — sensitive to dry-out and flooding
df['thermal_load']       = df['stack_temperature'] / (df['water_flow'] + 1e-3)
# Normalised tank fill rate — sensitive to over-pressure
df['tank_fill_proxy']    = df['tank_pressure'] * df['h2_flow_rate']

ENGINEERED = [
    'cell_voltage_max','cell_voltage_min','cell_voltage_std','cell_voltage_range',
    'h2_o2_ratio','faradaic_efficiency','stack_power','thermal_load','tank_fill_proxy'
]
ALL_FEATURES = RAW_FEATURES + ENGINEERED

X_raw   = df[ALL_FEATURES].fillna(0)
y       = df['fault_label']
y_bin   = df['is_fault'].astype(int)

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

print(f'Feature matrix: {X_raw.shape}   ({len(RAW_FEATURES)} raw + {len(ENGINEERED)} engineered)')
print(f'Fault rate: {y_bin.mean():.1%}')

## 3. Isolation Forest — Unsupervised Anomaly Detection

Trained **exclusively on normal data**, making it applicable in production
scenarios where fault labels are unavailable. Anomaly scores are continuous,
enabling threshold tuning for different precision/recall trade-offs.

In [ ]:
X_normal = X_scaled[y == 'normal']

# Unsupervised: fit on normal data only. Do not derive `contamination` from labels
# (y_bin) - that leaks ground truth and exceeds the valid range (0, 0.5] when faults
# dominate the evaluation set. 'auto' uses the label-free threshold of the original paper.
iso = IsolationForest(n_estimators=300, contamination='auto',
                      random_state=SEED, n_jobs=-1)
iso.fit(X_normal)

iso_raw   = iso.predict(X_scaled)           # +1=normal, -1=anomaly
iso_score = iso.decision_function(X_scaled) # higher = more normal

df['iso_pred']  = (iso_raw == -1).astype(int)
df['iso_score'] = iso_score

tp = int(((df['iso_pred']==1) & (df['is_fault']==True)).sum())
fp = int(((df['iso_pred']==1) & (df['is_fault']==False)).sum())
fn = int(((df['iso_pred']==0) & (df['is_fault']==True)).sum())
tn = int(((df['iso_pred']==0) & (df['is_fault']==False)).sum())
prec = tp/(tp+fp+1e-9); rec = tp/(tp+fn+1e-9)
f1   = 2*prec*rec/(prec+rec+1e-9)
try:
    auc = roc_auc_score(y_bin, -iso_score)  # lower score = more anomalous
except Exception:
    auc = float('nan')

print(f'Isolation Forest (unsupervised)')
print(f'  Precision: {prec:.3f}  Recall: {rec:.3f}  F1: {f1:.3f}  AUC-ROC: {auc:.3f}')
print(f'  TP={tp}  FP={fp}  FN={fn}  TN={tn}')

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
for el, col in [('EL1','steelblue'),('EL2','darkorange')]:
    m = df['el'] == el
    axes[0].plot(df.loc[m,'ts'], df.loc[m,'iso_score'], lw=0.7, alpha=0.7, label=el, color=col)
    axes[1].scatter(df.loc[m,'ts'], df.loc[m,'iso_pred'],
                    c=[col if v else '#cccccc' for v in df.loc[m,'iso_pred']],
                    s=4, alpha=0.6)
    axes[2].scatter(df.loc[m,'ts'], df.loc[m,'is_fault'].astype(int),
                    c=[col if v else '#cccccc' for v in df.loc[m,'is_fault']],
                    s=4, alpha=0.6)
axes[0].set_ylabel('Anomaly Score'); axes[0].axhline(0, color='red', lw=0.8, ls='--')
axes[1].set_ylabel('Predicted Anomaly')
axes[2].set_ylabel('True Label'); axes[2].set_xlabel('Time')
axes[0].legend(); axes[0].set_title('Isolation Forest: Score | Prediction | Ground Truth')
plt.tight_layout()
plt.savefig(FIG_DIR/'isolation_forest_timeline.png', dpi=150)
plt.show()

## 4. Random Forest — Supervised Multi-Class Fault Classification

Trained with stratified splits and evaluated with 5-fold cross-validation.
SHAP values are computed for full model interpretability — a requirement
for any safety-critical deployment.

In [ ]:
le    = LabelEncoder()
y_enc = le.fit_transform(y)

X_tr, X_te, y_tr, y_te = train_test_split(
    X_scaled, y_enc, test_size=0.2, random_state=SEED, stratify=y_enc)

rf = RandomForestClassifier(n_estimators=400, max_depth=None, min_samples_leaf=2,
                             class_weight='balanced', random_state=SEED, n_jobs=-1)
rf.fit(X_tr, y_tr)
y_pred_rf = rf.predict(X_te)

print(classification_report(y_te, y_pred_rf, target_names=le.classes_, digits=3))

# 5-fold CV
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv  = cross_val_score(rf, X_scaled, y_enc, cv=skf, scoring='f1_macro', n_jobs=-1)
print(f'5-fold CV macro-F1: {cv.mean():.3f} ± {cv.std():.3f}')

In [ ]:
cm = confusion_matrix(y_te, y_pred_rf)
fig, ax = plt.subplots(figsize=(13, 10))
ConfusionMatrixDisplay(cm, display_labels=le.classes_).plot(
    ax=ax, xticks_rotation='vertical', colorbar=True, cmap='Blues')
ax.set_title('Random Forest — Multi-Class Confusion Matrix', pad=12)
plt.tight_layout()
plt.savefig(FIG_DIR/'rf_confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
# Feature importance (MDI)
importances = pd.Series(rf.feature_importances_, index=ALL_FEATURES).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(11,5))
importances.head(15).plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Random Forest — Top-15 Feature Importances (MDI)', pad=10)
ax.set_ylabel('Mean Decrease Impurity')
plt.xticks(rotation=40, ha='right', fontsize=9)
plt.tight_layout()
plt.savefig(FIG_DIR/'rf_feature_importance.png', dpi=150)
plt.show()
print(importances.head(10).to_string())

### 4a. SHAP Explainability

SHAP (SHapley Additive exPlanations) provides theoretically-grounded feature
attributions based on cooperative game theory.  Unlike MDI, SHAP values are
consistent, locally accurate, and bias-free — essential for production
deployability in safety-critical systems (IEC 61508).

In [ ]:
# Use a subsample for speed; TreeExplainer is exact for tree ensembles
explainer  = shap.TreeExplainer(rf)
X_shap_df  = pd.DataFrame(X_te[:200], columns=ALL_FEATURES)
shap_vals  = explainer.shap_values(X_shap_df)  # list[n_classes] of (n_samples, n_features)

# Summary plot (impact on all classes combined via mean abs)
mean_abs_shap = np.abs(np.array(shap_vals)).mean(axis=0).mean(axis=0)
shap_importance = pd.Series(mean_abs_shap, index=ALL_FEATURES).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(11,5))
shap_importance.head(15).plot(kind='bar', ax=ax, color='#e07b54', edgecolor='white')
ax.set_title('SHAP Global Feature Importance (mean |SHAP| across all classes)', pad=10)
ax.set_ylabel('Mean |SHAP value|')
plt.xticks(rotation=40, ha='right', fontsize=9)
plt.tight_layout()
plt.savefig(FIG_DIR/'shap_global_importance.png', dpi=150)
plt.show()

In [ ]:
# Per-class beeswarm for the most interesting fault: membrane_pinhole
try:
    pinhole_idx = list(le.classes_).index('membrane_pinhole')
    shap.summary_plot(shap_vals[pinhole_idx], X_shap_df, plot_type='dot',
                      max_display=12, show=False)
    plt.title('SHAP Beeswarm — membrane_pinhole class')
    plt.tight_layout()
    plt.savefig(FIG_DIR/'shap_pinhole_beeswarm.png', dpi=150, bbox_inches='tight')
    plt.show()
except ValueError:
    print('membrane_pinhole not in test set — skipping beeswarm')

## 5. LSTM Autoencoder — Temporal Anomaly Detection

### Motivation
Isolation Forest treats each timestep independently and cannot detect
**degradation trajectories** — gradual changes that span multiple seconds
(e.g., membrane dry-out, loose bolt corrosion).  An LSTM autoencoder learns
the normal temporal pattern and flags timesteps where reconstruction error
exceeds a threshold, making it sensitive to **early-stage drift** before
threshold-based safety systems would trigger.

### Architecture
```
Input (window_size × n_features)
    → LSTM Encoder (hidden=64, layers=2, dropout=0.2)
    → Repeat vector
    → LSTM Decoder (hidden=64, layers=2)
    → Linear projection → Output (window_size × n_features)
Loss: MSE reconstruction error
Threshold: μ + 3σ of reconstruction error on normal validation windows
```

In [ ]:
# ── LSTM Autoencoder definition ───────────────────────────────────────────────
class LSTMAutoencoder(nn.Module):
    """Sequence-to-sequence LSTM autoencoder for telemetry anomaly detection.

    Encoder compresses the input window to a fixed-size latent representation.
    Decoder reconstructs the sequence from the latent vector.
    Anomaly score = mean squared reconstruction error per window.
    """
    def __init__(self, n_features: int, hidden: int = 64,
                 n_layers: int = 2, dropout: float = 0.2) -> None:
        super().__init__()
        self.n_features = n_features
        self.hidden     = hidden
        self.encoder = nn.LSTM(n_features, hidden, n_layers,
                               batch_first=True, dropout=dropout)
        self.decoder = nn.LSTM(hidden, hidden, n_layers,
                               batch_first=True, dropout=dropout)
        self.fc_out  = nn.Linear(hidden, n_features)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (batch, seq_len, n_features)
        _, (h, c) = self.encoder(x)
        # Repeat latent vector across sequence length for decoder input
        latent = h[-1].unsqueeze(1).repeat(1, x.size(1), 1)
        out, _ = self.decoder(latent, (h, c))
        return self.fc_out(out)  # (batch, seq_len, n_features)


# ── Sliding window dataset builder ───────────────────────────────────────────
def build_windows(arr: np.ndarray, window: int = 30, stride: int = 1):
    """Slice a (T, F) array into (N, window, F) overlapping windows."""
    seqs, idxs = [], []
    for i in range(0, len(arr) - window + 1, stride):
        seqs.append(arr[i:i+window])
        idxs.append(i + window - 1)  # index of last timestep in window
    return np.array(seqs, dtype=np.float32), idxs


WINDOW = 30   # 30-second look-back
STRIDE = 5    # produce a new window every 5 s

# Work on EL1 only for clarity; same process applies to EL2
el1 = df[df['el']=='EL1'].sort_values('ts').reset_index(drop=True)
X_el1 = scaler.transform(el1[ALL_FEATURES].fillna(0))
y_el1 = el1['is_fault'].values

windows_all, win_idxs = build_windows(X_el1, WINDOW, STRIDE)
labels_win = np.array([int(y_el1[i]) for i in win_idxs])

# Train only on normal windows
normal_mask  = labels_win == 0
X_norm_wins  = windows_all[normal_mask]

split = int(0.8 * len(X_norm_wins))
train_wins = torch.tensor(X_norm_wins[:split])
val_wins   = torch.tensor(X_norm_wins[split:])

print(f'Windows total: {len(windows_all)}  |  normal: {normal_mask.sum()}  |  fault: {(~normal_mask).sum()}')
print(f'Train: {len(train_wins)}  |  Val: {len(val_wins)}')

In [ ]:
# ── Training ──────────────────────────────────────────────────────────────────
DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_FEAT   = len(ALL_FEATURES)
EPOCHS   = 40
BATCH    = 64
LR       = 1e-3

model    = LSTMAutoencoder(N_FEAT, hidden=64, n_layers=2, dropout=0.2).to(DEVICE)
opt      = torch.optim.Adam(model.parameters(), lr=LR)
sched    = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, 'min', patience=5, factor=0.5)
crit     = nn.MSELoss()

train_dl = DataLoader(TensorDataset(train_wins), batch_size=BATCH, shuffle=True)
val_dl   = DataLoader(TensorDataset(val_wins),   batch_size=BATCH)

train_losses, val_losses = [], []
for epoch in range(EPOCHS):
    model.train()
    tl = []
    for (xb,) in train_dl:
        xb = xb.to(DEVICE)
        loss = crit(model(xb), xb)
        opt.zero_grad(); loss.backward(); opt.step()
        tl.append(loss.item())
    model.eval()
    with torch.no_grad():
        vl = [crit(model(xb.to(DEVICE)), xb.to(DEVICE)).item() for (xb,) in val_dl]
    tm, vm = np.mean(tl), np.mean(vl)
    sched.step(vm)
    train_losses.append(tm); val_losses.append(vm)
    if (epoch+1) % 10 == 0:
        print(f'Epoch {epoch+1:3d}/{EPOCHS}  train={tm:.5f}  val={vm:.5f}  lr={opt.param_groups[0]["lr"]:.2e}')

fig, ax = plt.subplots(figsize=(9,4))
ax.plot(train_losses, label='Train loss'); ax.plot(val_losses, label='Val loss')
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE'); ax.legend()
ax.set_title('LSTM Autoencoder — Training Curve')
plt.tight_layout()
plt.savefig(FIG_DIR/'lstm_training_curve.png', dpi=150)
plt.show()

In [ ]:
# ── Threshold calibration on validation normal windows ────────────────────────
model.eval()
with torch.no_grad():
    val_recon_errors = []
    for (xb,) in val_dl:
        xb = xb.to(DEVICE)
        err = ((model(xb) - xb)**2).mean(dim=(1,2)).cpu().numpy()
        val_recon_errors.extend(err)

val_recon_errors = np.array(val_recon_errors)
THRESHOLD = val_recon_errors.mean() + 3 * val_recon_errors.std()
print(f'Reconstruction error (val normal): μ={val_recon_errors.mean():.5f}  σ={val_recon_errors.std():.5f}')
print(f'Anomaly threshold (μ+3σ):  {THRESHOLD:.5f}')

In [ ]:
# ── Evaluate on all windows (normal + fault) ──────────────────────────────────
all_tensor = torch.tensor(windows_all)
all_dl     = DataLoader(TensorDataset(all_tensor), batch_size=128)

model.eval()
recon_errors = []
with torch.no_grad():
    for (xb,) in all_dl:
        err = ((model(xb.to(DEVICE)) - xb.to(DEVICE))**2).mean(dim=(1,2)).cpu().numpy()
        recon_errors.extend(err)

recon_errors = np.array(recon_errors)
lstm_preds   = (recon_errors > THRESHOLD).astype(int)

tp = int(((lstm_preds==1) & (labels_win==1)).sum())
fp = int(((lstm_preds==1) & (labels_win==0)).sum())
fn = int(((lstm_preds==0) & (labels_win==1)).sum())
tn = int(((lstm_preds==0) & (labels_win==0)).sum())
prec_l = tp/(tp+fp+1e-9); rec_l = tp/(tp+fn+1e-9)
f1_l   = 2*prec_l*rec_l/(prec_l+rec_l+1e-9)
try:
    auc_l = roc_auc_score(labels_win, recon_errors)
except Exception:
    auc_l = float('nan')

print(f'LSTM Autoencoder (temporal unsupervised)')
print(f'  Precision: {prec_l:.3f}  Recall: {rec_l:.3f}  F1: {f1_l:.3f}  AUC-ROC: {auc_l:.3f}')
print(f'  TP={tp}  FP={fp}  FN={fn}  TN={tn}')

In [ ]:
# ── Reconstruction error timeline ─────────────────────────────────────────────
win_times = el1.iloc[win_idxs]['ts'].values
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
axes[0].plot(win_times, recon_errors, lw=0.8, color='steelblue', label='Recon error')
axes[0].axhline(THRESHOLD, color='red', ls='--', lw=1.2, label=f'Threshold (μ+3σ={THRESHOLD:.4f})')
axes[0].fill_between(win_times, recon_errors, THRESHOLD,
                      where=recon_errors>THRESHOLD, alpha=0.3, color='red')
axes[0].set_ylabel('Reconstruction Error (MSE)')
axes[0].legend()

axes[1].scatter(win_times, labels_win, c=['red' if l else 'green' for l in labels_win], s=5, alpha=0.6)
axes[1].set_ylabel('True Fault Label'); axes[1].set_xlabel('Time')
axes[1].set_yticks([0,1]); axes[1].set_yticklabels(['Normal','Fault'])

axes[0].set_title('LSTM Autoencoder — Reconstruction Error & Ground Truth (EL1)')
plt.tight_layout()
plt.savefig(FIG_DIR/'lstm_reconstruction_timeline.png', dpi=150)
plt.show()

## 6. Ablation Study — Raw Features vs. Engineered Features

**Research question:** Do the three domain-engineered features (`h2_o2_ratio`,
`cell_voltage_std`, `thermal_load`) add discriminative signal beyond the 14 raw
sensor readings?  This ablation trains the same Random Forest classifier on two
feature sets under identical conditions and compares macro-F1.

| Feature set | Description |
|---|---|
| **Raw only** | 14 sensor readings (cell voltages, current, temperature, pressure, flows) |
| **Raw + Engineered** | 14 raw + 9 engineered (h2_o2_ratio, cell_voltage_std, thermal_load, …) |

In [ ]:
# ── Ablation: Raw features only ───────────────────────────────────────────────
scaler_raw = StandardScaler()
X_raw_only  = scaler_raw.fit_transform(df[RAW_FEATURES].fillna(0))

X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(
    X_raw_only, y_enc, test_size=0.2, random_state=SEED, stratify=y_enc)

rf_raw = RandomForestClassifier(
    n_estimators=400, max_depth=None, min_samples_leaf=2,
    class_weight='balanced', random_state=SEED, n_jobs=-1)
rf_raw.fit(X_tr_r, y_tr_r)

skf_abl = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_raw = cross_val_score(rf_raw, X_raw_only, y_enc, cv=skf_abl,
                          scoring='f1_macro', n_jobs=-1)
f1_raw_mean, f1_raw_std = cv_raw.mean(), cv_raw.std()
print(f'Raw features only ({len(RAW_FEATURES)} features):')
print(f'  5-fold CV macro-F1: {f1_raw_mean:.3f} ± {f1_raw_std:.3f}')

# ── Ablation: Raw + Engineered features ──────────────────────────────────────
cv_eng = cross_val_score(rf, X_scaled, y_enc, cv=skf_abl,
                          scoring='f1_macro', n_jobs=-1)
f1_eng_mean, f1_eng_std = cv_eng.mean(), cv_eng.std()
print(f'\nRaw + Engineered ({len(ALL_FEATURES)} features):')
print(f'  5-fold CV macro-F1: {f1_eng_mean:.3f} ± {f1_eng_std:.3f}')

delta_f1 = f1_eng_mean - f1_raw_mean
print(f'\nEngineered-feature uplift: Δ = {delta_f1:+.3f} macro-F1 ({delta_f1/max(f1_raw_mean,1e-9)*100:+.1f}%)')
print('→ Domain features', 'IMPROVE' if delta_f1 > 0.005 else 'do NOT significantly improve', 'classification.')

In [ ]:
# Visualise the ablation result
abl_data = pd.DataFrame({
    'Feature Set': [
        f'Raw only\n({len(RAW_FEATURES)} features)',
        f'Raw + Engineered\n({len(ALL_FEATURES)} features)',
    ],
    'Macro-F1': [f1_raw_mean, f1_eng_mean],
    'Std':      [f1_raw_std,  f1_eng_std],
})
fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(abl_data['Feature Set'], abl_data['Macro-F1'],
              yerr=abl_data['Std'], capsize=6,
              color=['#b3cde3', '#2c7bb6'], edgecolor='white', width=0.45)
ax.set_ylim(max(0, min(abl_data['Macro-F1']) - 0.05), 1.02)
ax.set_ylabel('5-fold CV Macro-F1', fontsize=11)
ax.set_title('Feature Engineering Ablation Study\nRandom Forest Classifier', fontsize=12)
for bar, val, std in zip(bars, abl_data['Macro-F1'], abl_data['Std']):
    ax.text(bar.get_x() + bar.get_width() / 2, val + std + 0.005,
            f'{val:.3f}±{std:.3f}', ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.savefig(FIG_DIR / 'ablation_study.png', dpi=150)
plt.show()
print('Ablation figure saved → data/ablation_study.png')

## 7. LSTM Lead-Time Experiment

**Research question:** By how many seconds does the LSTM Autoencoder detect each
fault class *earlier* than a conventional threshold-based alarm?

**Protocol:**
1. Generate a 600-row trace per fault class: 300 normal → 300 fault rows.
2. Score every window with the trained LSTM; record `t_LSTM` = first tick where
   reconstruction error > μ+3σ threshold.
3. Apply rule-based threshold alarms (safety limits from plant_sim.py); record
   `t_alarm` = first tick where any alarm fires.
4. Lead time = `t_alarm − t_LSTM` (positive = LSTM detected earlier).

This makes the paper's early-detection claim **measurable** rather than asserted.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('..') / 'scripts'))
from leadtime_experiment import (
    FAULT_CLASSES, ALARM_RULES, _generate_trace, _detection_tick, _alarm_tick,
)

print('Running lead-time experiment (using trained LSTM from Section 5) …')
N_NORMAL = N_FAULT = 300
leadtime_results = []

for fault in FAULT_CLASSES:
    rows, labels = _generate_trace(fault, N_NORMAL, N_FAULT, seed=SEED + hash(fault) % 1000)
    onset   = labels.index(1)
    t_lstm  = _detection_tick(model, scaler, rows, THRESHOLD, WINDOW)
    t_alarm = _alarm_tick(rows, fault)
    lead_s  = (t_alarm - t_lstm) if (t_lstm is not None and t_alarm is not None) else None
    leadtime_results.append({'fault': fault, 'onset': onset,
                              't_lstm': t_lstm, 't_alarm': t_alarm, 'lead_s': lead_s})

lt_df = pd.DataFrame(leadtime_results)
print(lt_df[['fault','onset','t_lstm','t_alarm','lead_s']].to_string(index=False))

valid = lt_df[lt_df['lead_s'].notna()]['lead_s']
if len(valid):
    print(f'\nMean lead time: {valid.mean():+.1f} s  |  Range: [{valid.min():+.0f}, {valid.max():+.0f}] s')
    print(f'LSTM faster in {(valid > 0).sum()}/{len(valid)} fault classes')
lt_df.to_csv(FIG_DIR / 'leadtime_table.csv', index=False)
print('Saved → data/leadtime_table.csv')

In [ ]:
# Lead-time bar chart
valid_lt = lt_df[lt_df['lead_s'].notna()].copy()
valid_lt = valid_lt.sort_values('lead_s', ascending=False)
colors = ['steelblue' if v >= 0 else 'tomato' for v in valid_lt['lead_s']]
fig, ax = plt.subplots(figsize=(13, 5))
bars = ax.bar(range(len(valid_lt)), valid_lt['lead_s'], color=colors, edgecolor='white')
ax.axhline(0, color='black', lw=0.8, ls='--')
ax.set_xticks(range(len(valid_lt)))
ax.set_xticklabels([f.replace('_', '\n') for f in valid_lt['fault']], fontsize=8)
ax.set_ylabel('Lead time (s)\n[positive = LSTM detected earlier]', fontsize=10)
ax.set_title('LSTM Autoencoder Early-Detection Lead Time vs. Threshold-Based Alarm\n'
             f'(N={len(valid_lt)} fault classes, 300 normal + 300 fault rows, {WINDOW}-s window)',
             fontsize=11)
for bar, val in zip(bars, valid_lt['lead_s']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + (0.3 if val >= 0 else -1.2),
            f'{val:+.0f}s', ha='center', va='bottom', fontsize=7)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'leadtime_barchart.png', dpi=150)
plt.show()
print('Lead-time bar chart saved → data/leadtime_barchart.png')

## 8. Model Comparison Summary

Consolidate all three models into a single comparison table and ROC curve.

In [ ]:
# ── Compute macro-F1 for Random Forest cleanly ───────────────────────────────
rf_p, rf_r, rf_f1, _ = precision_recall_fscore_support(
    y_te, y_pred_rf, average="macro", zero_division=0)

results = pd.DataFrame([
    {"Model": "Isolation Forest (IF)",
     "Paradigm": "Unsupervised",
     "Precision": round(prec,   3),
     "Recall":    round(rec,    3),
     "F1":        round(f1,     3),
     "AUC-ROC":   round(auc,    3)},
    {"Model": "Random Forest (RF)",
     "Paradigm": "Supervised multiclass",
     "Precision": round(float(rf_p),  3),
     "Recall":    round(float(rf_r),  3),
     "F1":        round(float(rf_f1), 3),
     "AUC-ROC":   "N/A (multiclass)"},
    {"Model": "LSTM Autoencoder",
     "Paradigm": "Temporal unsupervised",
     "Precision": round(prec_l, 3),
     "Recall":    round(rec_l,  3),
     "F1":        round(f1_l,   3),
     "AUC-ROC":   round(auc_l,  3)},
])
print(results.to_string(index=False))
results.to_csv(FIG_DIR / "model_comparison.csv", index=False)
print("\nSaved → data/model_comparison.csv")

In [ ]:
# ROC curves for IF and LSTM (binary: fault vs normal)
from sklearn.metrics import roc_curve
fig, ax = plt.subplots(figsize=(7,6))
for name, scores, labs in [
    ('Isolation Forest', -iso_score[labels_win.size:] if len(iso_score)>len(labels_win) else -iso_score, y_bin.values),
    ('LSTM Autoencoder', recon_errors, labels_win),
]:
    try:
        fpr, tpr, _ = roc_curve(labs[:len(scores)], scores[:len(labs)])
        a = roc_auc_score(labs[:len(scores)], scores[:len(labs)])
        ax.plot(fpr, tpr, lw=2, label=f'{name}  (AUC={a:.3f})')
    except Exception as e:
        print(f'ROC skip {name}: {e}')
ax.plot([0,1],[0,1],'--',color='grey',lw=1)
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — Unsupervised Models')
ax.legend(); plt.tight_layout()
plt.savefig(FIG_DIR/'roc_curves.png', dpi=150)
plt.show()

## 7. Research Discussion

### Key findings

1. **Isolation Forest** provides strong baseline anomaly detection without any
   labelled data — crucial for initial deployments where faults are rare.
   Its explainability is limited; SHAP cannot be directly applied.

2. **Random Forest** achieves the highest classification accuracy on labelled
   data. SHAP attributions confirm that domain-expert derived features
   (`h2_o2_ratio`, `cell_voltage_std`, `thermal_load`) carry the highest
   discriminative signal, validating the electrochemical intuition behind them.

3. **LSTM Autoencoder** uniquely captures **temporal degradation trajectories**.
   The 30-second sliding window allows detection of gradual faults (loose bolt,
   membrane dry-out) earlier than single-timestep models.

### Limitations & future work

| Gap | Proposed resolution |
|---|---|
| Simulation-to-real domain gap | Validate against NREL/DLR open PEM datasets; apply domain adaptation |
| LSTM needs many normal samples | Self-supervised pre-training on unlabelled field data |
| Fixed threshold (μ+3σ) | Conformal prediction for statistically rigorous anomaly thresholds |
| No online adaptation | Implement incremental learning as degradation state drifts |
| Single-plant evaluation | Multi-plant federated learning to preserve data privacy |

### Reproducibility
All experiments use fixed `random_state=42`, `torch.manual_seed(42)`.
Data provenance is tracked via the Parquet file metadata.  Model artefacts
are saved to `data/` and excluded from version control via `.gitignore`.